# Implement DDPM (Denoising Diffusion) from Scratch - SOLUTION

**Difficulty**: 🔴 Hard

**Companies**: Midjourney, Runway, Stability AI, Adobe, Google

---

### Problem Statement

Denoising Diffusion Probabilistic Models (DDPMs) generate data by learning to reverse a gradual noising process. The **forward process** adds Gaussian noise to data over T timesteps until it becomes pure noise. The **reverse process** learns to denoise step by step, recovering the original data distribution.

Your task is to implement the complete DDPM pipeline: the noise schedule, the forward noising process, a simple denoiser network, and the reverse sampling loop. You will train on a simple 2D distribution (concentric circles) and generate new samples.

---

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_circles

In [ ]:
# Generate training data: concentric circles
torch.manual_seed(42)
np.random.seed(42)

n_samples = 2000
data, _ = make_circles(n_samples=n_samples, noise=0.05, factor=0.5)
data = torch.tensor(data, dtype=torch.float32)

# Visualize
plt.figure(figsize=(5, 5))
plt.scatter(data[:, 0].numpy(), data[:, 1].numpy(), s=5, alpha=0.5)
plt.title('Training Data: Concentric Circles')
plt.axis('equal')
plt.show()

print(f'Data shape: {data.shape}')

In [ ]:
# --- Part 1: Linear Beta Schedule ---

def linear_beta_schedule(timesteps, beta_start=0.0001, beta_end=0.02):
    """
    Create a linear schedule of beta values for the forward diffusion process.

    Args:
        timesteps (int): Number of diffusion steps T.
        beta_start (float): Starting beta value.
        beta_end (float): Ending beta value.

    Returns:
        betas (Tensor): Shape (timesteps,) linearly spaced beta values.
    """
    return torch.linspace(beta_start, beta_end, timesteps)


# Precompute schedule quantities
T = 300
betas = linear_beta_schedule(T)

alphas = 1.0 - betas
alphas_cumprod = torch.cumprod(alphas, dim=0)
sqrt_alphas_cumprod = torch.sqrt(alphas_cumprod)
sqrt_one_minus_alphas_cumprod = torch.sqrt(1.0 - alphas_cumprod)

print(f'betas: min={betas.min():.6f}, max={betas.max():.6f}')
print(f'alphas_cumprod: first={alphas_cumprod[0]:.6f}, last={alphas_cumprod[-1]:.6f}')

In [ ]:
# --- Part 2: Forward Process (q_sample) ---

def extract(a, t, x_shape):
    """Extract values from schedule tensor `a` at timesteps `t`, reshape for broadcasting."""
    batch_size = t.shape[0]
    out = a.gather(-1, t)
    return out.reshape(batch_size, *((1,) * (len(x_shape) - 1)))


def q_sample(x_0, t, noise, sqrt_alphas_cumprod, sqrt_one_minus_alphas_cumprod):
    """
    Forward diffusion process: add noise to x_0 at timestep t.

    q(x_t | x_0) = N(x_t; sqrt(alpha_bar_t) * x_0, (1 - alpha_bar_t) * I)

    Args:
        x_0 (Tensor): Clean data, shape (B, D).
        t (Tensor): Timesteps, shape (B,).
        noise (Tensor): Gaussian noise, same shape as x_0.
        sqrt_alphas_cumprod (Tensor): Precomputed schedule values.
        sqrt_one_minus_alphas_cumprod (Tensor): Precomputed schedule values.

    Returns:
        x_t (Tensor): Noised data at timestep t.
    """
    sqrt_alpha_bar = extract(sqrt_alphas_cumprod, t, x_0.shape)
    sqrt_one_minus_alpha_bar = extract(sqrt_one_minus_alphas_cumprod, t, x_0.shape)
    return sqrt_alpha_bar * x_0 + sqrt_one_minus_alpha_bar * noise


# Visualize forward process at various timesteps
fig, axes = plt.subplots(1, 5, figsize=(20, 4))
timesteps_to_show = [0, 50, 100, 200, 299]
for ax, t_val in zip(axes, timesteps_to_show):
    t_batch = torch.full((n_samples,), t_val, dtype=torch.long)
    noise = torch.randn_like(data)
    x_noisy = q_sample(data, t_batch, noise, sqrt_alphas_cumprod, sqrt_one_minus_alphas_cumprod)
    ax.scatter(x_noisy[:, 0].numpy(), x_noisy[:, 1].numpy(), s=2, alpha=0.5)
    ax.set_title(f't = {t_val}')
    ax.set_xlim(-3, 3)
    ax.set_ylim(-3, 3)
    ax.set_aspect('equal')
plt.suptitle('Forward Diffusion Process')
plt.tight_layout()
plt.show()

In [ ]:
# --- Part 3: Denoiser Network ---

class SinusoidalPositionEmbeddings(nn.Module):
    """Sinusoidal embeddings for timestep encoding."""
    def __init__(self, dim):
        super().__init__()
        self.dim = dim

    def forward(self, t):
        device = t.device
        half_dim = self.dim // 2
        emb = np.log(10000) / (half_dim - 1)
        emb = torch.exp(torch.arange(half_dim, device=device) * -emb)
        emb = t[:, None].float() * emb[None, :]
        emb = torch.cat([emb.sin(), emb.cos()], dim=-1)
        return emb


class ResidualBlock(nn.Module):
    """Simple residual MLP block."""
    def __init__(self, dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim, dim),
            nn.GELU(),
            nn.Linear(dim, dim),
        )
        self.act = nn.GELU()

    def forward(self, x):
        return self.act(x + self.net(x))


class SimpleDenoiser(nn.Module):
    """
    Simple MLP-based denoiser that predicts noise given (x_t, t).
    """
    def __init__(self, data_dim=2, hidden_dim=128, time_dim=64):
        super().__init__()
        self.time_mlp = nn.Sequential(
            SinusoidalPositionEmbeddings(time_dim),
            nn.Linear(time_dim, time_dim),
            nn.GELU(),
        )

        self.input_proj = nn.Linear(data_dim, hidden_dim)
        self.time_proj = nn.Linear(time_dim, hidden_dim)

        self.net = nn.Sequential(
            ResidualBlock(hidden_dim),
            ResidualBlock(hidden_dim),
            ResidualBlock(hidden_dim),
        )

        self.output_proj = nn.Linear(hidden_dim, data_dim)

    def forward(self, x, t):
        """
        Predict noise added to x at timestep t.

        Args:
            x (Tensor): Noised input, shape (B, data_dim).
            t (Tensor): Timesteps, shape (B,).

        Returns:
            Tensor: Predicted noise, shape (B, data_dim).
        """
        t_emb = self.time_mlp(t)
        h = self.input_proj(x) + self.time_proj(t_emb)
        h = self.net(h)
        return self.output_proj(h)


model = SimpleDenoiser(data_dim=2, hidden_dim=128, time_dim=64)
print(f'Model parameters: {sum(p.numel() for p in model.parameters()):,}')

In [ ]:
# --- Part 4: Training Loop ---

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
n_epochs = 300
batch_size = 256
losses = []

model.train()
for epoch in range(n_epochs):
    perm = torch.randperm(n_samples)
    epoch_loss = 0.0
    n_batches = 0

    for i in range(0, n_samples, batch_size):
        x_0 = data[perm[i:i+batch_size]]
        b = x_0.shape[0]

        # Sample random timesteps
        t = torch.randint(0, T, (b,))

        # Sample noise
        noise = torch.randn_like(x_0)

        # Get noisy data
        x_t = q_sample(x_0, t, noise, sqrt_alphas_cumprod, sqrt_one_minus_alphas_cumprod)

        # Predict noise
        predicted_noise = model(x_t, t)

        # MSE loss between predicted and actual noise
        loss = F.mse_loss(predicted_noise, noise)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()
        n_batches += 1

    avg_loss = epoch_loss / n_batches
    losses.append(avg_loss)
    if (epoch + 1) % 50 == 0:
        print(f'Epoch {epoch+1}/{n_epochs}, Loss: {avg_loss:.6f}')

plt.plot(losses)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training Loss')
plt.show()

In [ ]:
# --- Part 5: Reverse Sampling Loop ---

@torch.no_grad()
def p_sample(model, x_t, t_index, betas, alphas, alphas_cumprod):
    """
    Single reverse diffusion step: sample x_{t-1} from x_t.

    Args:
        model: Trained denoiser.
        x_t (Tensor): Current noisy sample, shape (B, D).
        t_index (int): Current timestep index.
        betas, alphas, alphas_cumprod: Schedule tensors.

    Returns:
        x_{t-1} (Tensor): Denoised sample one step back.
    """
    b = x_t.shape[0]
    t_batch = torch.full((b,), t_index, dtype=torch.long)

    # Predict noise
    predicted_noise = model(x_t, t_batch)

    # Get schedule values for this timestep
    beta_t = betas[t_index]
    alpha_t = alphas[t_index]
    alpha_bar_t = alphas_cumprod[t_index]

    # Compute mean of p(x_{t-1} | x_t)
    # mu = (1 / sqrt(alpha_t)) * (x_t - (beta_t / sqrt(1 - alpha_bar_t)) * predicted_noise)
    mean = (1.0 / torch.sqrt(alpha_t)) * (
        x_t - (beta_t / torch.sqrt(1.0 - alpha_bar_t)) * predicted_noise
    )

    if t_index == 0:
        return mean
    else:
        noise = torch.randn_like(x_t)
        sigma = torch.sqrt(beta_t)
        return mean + sigma * noise


@torch.no_grad()
def p_sample_loop(model, shape, timesteps, betas, alphas, alphas_cumprod):
    """
    Full reverse diffusion sampling loop.

    Args:
        model: Trained denoiser.
        shape (tuple): Shape of samples to generate, e.g. (n_samples, 2).
        timesteps (int): Number of diffusion steps T.
        betas, alphas, alphas_cumprod: Schedule tensors.

    Returns:
        samples (Tensor): Generated samples, shape `shape`.
    """
    # Start from pure Gaussian noise
    x = torch.randn(shape)

    # Iteratively denoise from t = T-1 down to t = 0
    for t_index in reversed(range(timesteps)):
        x = p_sample(model, x, t_index, betas, alphas, alphas_cumprod)

    return x


# Generate samples
model.eval()
samples = p_sample_loop(model, shape=(1000, 2), timesteps=T, betas=betas, alphas=alphas, alphas_cumprod=alphas_cumprod)

print(f'Generated samples shape: {samples.shape}')

In [ ]:
# --- Validation ---

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].scatter(data[:, 0].numpy(), data[:, 1].numpy(), s=5, alpha=0.5)
axes[0].set_title('Training Data')
axes[0].set_xlim(-2, 2)
axes[0].set_ylim(-2, 2)
axes[0].set_aspect('equal')

axes[1].scatter(samples[:, 0].numpy(), samples[:, 1].numpy(), s=5, alpha=0.5, c='orange')
axes[1].set_title('Generated Samples (DDPM)')
axes[1].set_xlim(-2, 2)
axes[1].set_ylim(-2, 2)
axes[1].set_aspect('equal')

plt.tight_layout()
plt.show()

# Basic checks
print('=== Validation ===')
print(f'Training loss (final): {losses[-1]:.6f}')
assert losses[-1] < losses[0], 'Training loss should decrease!'
print('PASSED: Loss decreased during training')

# Check generated samples are in reasonable range
assert samples.abs().max() < 10, 'Generated samples have extreme values!'
print('PASSED: Generated samples in reasonable range')

# Check generated samples roughly match training distribution radius
train_radius = data.norm(dim=1).mean()
gen_radius = samples.norm(dim=1).mean()
print(f'Mean radius - Training: {train_radius:.3f}, Generated: {gen_radius:.3f}')
assert abs(train_radius - gen_radius) < 0.5, 'Generated distribution radius is too different!'
print('PASSED: Generated distribution roughly matches training distribution')

print('\nAll validations passed!')